# Setup

In [ ]:
%load_ext autoreload
%autoreload 2
from os import environ
from sys import path

from torch.backends import cudnn

# enforce more deterministic behavior
environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

path.append("..")

from processor.core.interaction_conductor.chat_interface import ChatInterface, ChatInterfaceOutputFormat

In [ ]:
USER_ID = "llm"
DATA_SOURCES = ["biomedical"]
INITIAL_PROMPT = """I’m interested in looking at tumor sample data from our recent studies—
could you show me what kinds of tumor types we have represented in the dataset? I want to get a sense of the distribution before narrowing down to anything specific."""

In [ ]:
llm_path = "model/weight/qwen3-8b"
embed_model_path = "model/weight/bge-base"
chat_interface = ChatInterface(llm_path, embed_model_path, USER_ID, DATA_SOURCES)

In [ ]:
from processor.core.ir_system.ir_data_model import convert_multi_retriever_results_to_str


def print_format_to_gpt(ci_output: ChatInterfaceOutputFormat):
    system_output = ci_output['system_response']
    state = ci_output['state']
    current_retrieval_results = ci_output['current_retrieval_results']

    print(f"""SYSTEM OUTPUT:
```{system_output}```

STATE:
```{state}```

RETRIEVED DATA BY THE SYSTEM:
```{convert_multi_retriever_results_to_str(current_retrieval_results)}```
""")

In [1]:
def print_initial_prompt_to_chatgpt(domain: str, question: str):
    print(f"""You are simulating a {domain} domain expert, who is interacting with a data assistant system to explore insights from an enterprise dataset. The system represents your information need as a set of target schemas, representing relevant table(s) for your question, along with a list of SQL statements, which if runs sequentially on the (materialized) target schemas, will result in the answer of your question.

In this scenario, the system already has access to internal environment-related dataset. You (the simulated user) are already somewhat familiar with the topics of the dataset, as it is commonly used in your team or organization. You are not uploading a new dataset or asking about the existence of some dataset. Your task is to gradually explore or refine your information need about some aspect of the data. You do not begin with a precise question; rather, your curiosity evolves based on system responses and your domain expertise.

Here is a possible eventual goal (you do not know this yet, but may arrive at it through exploration):

{question}

Your behavior should reflect the following:
- You are familiar with the domain.
- You explore and refine your question step-by-step depending on the system's ability to surface relevant information.
- You are allowed to be vague, get sidetracked, or go in the wrong direction.
- You will only arrive at the specific question above if the system's output correctly leads you there.

Start the conversation as the user. Your first turn should reflect a vague curiosity grounded in your domain knowledge, not a generic or unaware greeting.""")

print_initial_prompt_to_chatgpt(
    "biomedical", "What is the average age of patients with serous tumor samples analyzed in the study?"
)

You are simulating a biomedical domain expert, who is interacting with a data assistant system to explore insights from an enterprise dataset. The system represents your information need as a set of target schemas, representing relevant table(s) for your question, along with a list of SQL statements, which if runs sequentially on the (materialized) target schemas, will result in the answer of your question.

In this scenario, the system already has access to internal environment-related dataset. You (the simulated user) are already somewhat familiar with the topics of the dataset, as it is commonly used in your team or organization. You are not uploading a new dataset or asking about the existence of some dataset. Your task is to gradually explore or refine your information need about some aspect of the data. You do not begin with a precise question; rather, your curiosity evolves based on system responses and your domain expertise.

Here is a possible eventual goal (you do not know th

# INTERACTION

In [ ]:
output = chat_interface.process_user_input(INITIAL_PROMPT)
print_format_to_gpt(output)

In [ ]:
output = chat_interface.process_user_input("""Yes, please go ahead and materialize the **tumor_samples** table so we can see the tumor type counts.""")
print_format_to_gpt(output)

In [ ]:
output = chat_interface.process_user_input("""Let’s take a closer look at the **serous** cases—could you show me more details about those samples? For instance, basic patient demographics or key clinical features.""")
print_format_to_gpt(output)